# 10 — V3: độ ổn định và lỗi validation

Nếu chưa có kết quả, notebook chạy kiểm định chéo trên train và phân tích validation. Cần chạy notebook 09 trước. Không đánh giá test.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image
root = Path.cwd()
if not (root / 'configs/experiments_v3.yaml').exists(): root = root.parent
import sys
sys.path.insert(0, str(root))
out = root / 'results/v3'
if not (out / 'metrics/cv_summary.json').exists():
    if not (root / 'results/v2/metrics/comparison.json').is_file():
        raise RuntimeError('Chưa có kết quả V2. Hãy chạy notebook 09 trước.')
    from src.pipeline_v3 import run
    run()
summary = json.loads((out / 'metrics/cv_summary.json').read_text())
display(pd.DataFrame(summary['summaries']))
display(Image(filename=str(out / 'figures/cv_comparison.png')))

## Chỉ số từng lớp và lỗi validation

Mỗi lớp validation có 10 ảnh nên recall thay đổi theo bước 0,1. Cặp có hướng là lớp thực tế → lớp dự đoán. Ảnh cho thấy hiện tượng; tác động của màu, ánh sáng, tỉ lệ, hình dạng và nền chỉ là giả thuyết khi chưa có thử nghiệm kiểm soát.

In [ ]:
for experiment in ['E4', 'E5', 'E6']:
    print(experiment, 'các lớp có F1 thấp nhất:')
    display(pd.read_csv(out / f'metrics/{experiment}_per_class.csv').sort_values(['f1','class_id']).head(10))
    print('Các cặp nhầm có hướng thường gặp:')
    display(pd.read_csv(out / f'metrics/{experiment}_confusion_pairs.csv').head(10))
display(Image(filename=str(out / 'figures/E6_per_class_f1.png')))
display(Image(filename=str(out / 'figures/E6_confusion_examples.png')))

## Quan sát từ các ví dụ validation

E6 có 22 lớp validation đạt F1 = 0, mỗi lớp có 10 ảnh. Các cặp nhầm nhiều nhất là 24 → 11 và 94 → 32, mỗi cặp bốn ảnh. Ảnh 6814, 6819, 6834 cho thấy hoa lớp 24 màu đỏ/hồng giữa lá xanh; ảnh 7336 và 7386 của lớp 94 khác về tỉ lệ và nền. Ảnh 4260 và 4296 lớp 18 khác về ánh sáng và cách cắt. Chưa có thử nghiệm kiểm soát để khẳng định nguyên nhân lỗi.

## Diễn giải và quy trình

Các tập train của các fold giao nhau; hyperparameter được chọn trên validation riêng. Độ lệch chuẩn giữa fold chỉ mô tả biến thiên, không phải khoảng tin cậy. Các seed PCA chỉ thay đổi thuật toán randomized. Trước một lần đánh giá test E6 cần được cho phép riêng, phải khóa model, tiền xử lý và quy tắc báo cáo; test V1 đã từng được quan sát.